# EDA - IELTS Writing Task 2 (Hugging Face)

Dataset: `chillies/IELTS-writing-task-2-evaluation`

Muc tieu:
- Kiem tra quality co ban (null/empty/outlier)
- Chuan hoa cot `band`
- Thong ke do dai prompt/essay/evaluation
- Kiem tra duplicate
- Kiem tra tinh template cua cot `evaluation`


## 0) Cai dependencies (neu can)


In [ ]:
import sys
!{sys.executable} -m pip install -U datasets pandas numpy matplotlib seaborn


## 1) Load dataset


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import hashlib
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 180)

DATASET_NAME = 'chillies/IELTS-writing-task-2-evaluation'
ds = load_dataset(DATASET_NAME)
print(ds)

split_name = list(ds.keys())[0]
df = ds[split_name].to_pandas()
print('split:', split_name)
print('shape:', df.shape)
print('columns:', list(df.columns))

df.head(3)


## 2) Kiem tra null/empty + thong ke do dai


In [ ]:
expected_cols = ['prompt', 'essay', 'evaluation', 'band']
missing_cols = [c for c in expected_cols if c not in df.columns]
print('missing_cols:', missing_cols)

null_rates = df[expected_cols].isna().mean().sort_values(ascending=False)
print('\nNull rate:')
print(null_rates)

for c in expected_cols:
    empty_rate = df[c].astype(str).str.strip().eq('').mean()
    print(f'empty_rate[{c}] = {empty_rate:.4f}')

def wc(x):
    return len(str(x).split())

df['prompt_words'] = df['prompt'].map(wc)
df['essay_words'] = df['essay'].map(wc)
df['evaluation_words'] = df['evaluation'].map(wc)

display(df[['prompt_words','essay_words','evaluation_words']].describe(percentiles=[0.5,0.9,0.95,0.99]).T)


## 3) Parse va phan tich cot band


In [ ]:
band_re = re.compile(r'(\d(?:\.\d)?)')

def parse_band(x):
    if x is None:
        return np.nan
    s = str(x).strip()
    m = band_re.search(s)
    if not m:
        return np.nan
    try:
        return float(m.group(1))
    except Exception:
        return np.nan

df['band_value'] = df['band'].map(parse_band)
print('band parse null rate:', df['band_value'].isna().mean())
print('unique band values:', sorted(df['band_value'].dropna().unique().tolist()))

band_counts = df['band_value'].value_counts(dropna=False).sort_index()
display(band_counts)

plt.figure(figsize=(10,4))
sns.histplot(df['band_value'].dropna(), bins=20)
plt.title('Band distribution')
plt.xlabel('band')
plt.ylabel('count')
plt.show()


## 4) Duplicate check


In [ ]:
def norm_text(s):
    s = str(s)
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

def sha256_text(s):
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

df['prompt_hash'] = df['prompt'].map(norm_text).map(sha256_text)
df['essay_hash'] = df['essay'].map(norm_text).map(sha256_text)

n = len(df)
prompt_unique = df['prompt_hash'].nunique()
essay_unique = df['essay_hash'].nunique()
pair_unique = df[['prompt_hash','essay_hash']].drop_duplicates().shape[0]

print('rows:', n)
print('prompt dup rate:', round(1 - prompt_unique/n, 4))
print('essay dup rate:', round(1 - essay_unique/n, 4))
print('pair dup rate:', round(1 - pair_unique/n, 4))

print('\nTop duplicate prompts:')
display(df['prompt_hash'].value_counts().head(10))

print('Top duplicate essays:')
display(df['essay_hash'].value_counts().head(10))


## 5) Evaluation template check (heuristic)


In [ ]:
markers = [
    'Task Achievement', 'Task Response',
    'Coherence and Cohesion',
    'Lexical Resource',
    'Grammatical Range and Accuracy',
    'Overall Band Score',
    'Strengths', 'Areas for Improvement', 'Suggestions', 'Additional Comments'
]

presence = {}
ev = df['evaluation'].astype(str)
for m in markers:
    presence[m] = ev.str.contains(m, case=False, na=False).mean()

presence_df = pd.DataFrame({'marker': list(presence.keys()), 'presence_rate': list(presence.values())})\
    .sort_values('presence_rate', ascending=False)
display(presence_df)

first200 = ev.str.replace(r'\s+', ' ', regex=True).str.strip().str[:200]
fp_counts = first200.value_counts()
print('unique first-200 fingerprints:', fp_counts.shape[0])
display(fp_counts.head(10))

first_line = ev.str.splitlines().map(lambda xs: next((x.strip() for x in xs if x.strip()), ''))
display(first_line.value_counts().head(15))


## 6) Trich sub-scores (TR/CC/LR/GRA) tu evaluation (optional)


In [ ]:
def extract_score(text, aliases):
    t = str(text)
    for alias in aliases:
        patterns = [
            rf'{alias}[^\d]*\[\s*(\d(?:\.\d)?)\s*\]',
            rf'{alias}[^\d]*(\d(?:\.\d)?)'
        ]
        for p in patterns:
            m = re.search(p, t, flags=re.IGNORECASE)
            if m:
                try:
                    return float(m.group(1))
                except Exception:
                    pass
    return np.nan

aliases = {
    'TR': ['Task Achievement', 'Task Response'],
    'CC': ['Coherence and Cohesion', 'Coherence & Cohesion'],
    'LR': ['Lexical Resource', 'Vocabulary'],
    'GRA': ['Grammatical Range and Accuracy', 'Grammar']
}

for k, v in aliases.items():
    df[f'{k}_score'] = df['evaluation'].map(lambda x: extract_score(x, v))

display(df[['TR_score','CC_score','LR_score','GRA_score']].describe().T)
display(df[['band_value','TR_score','CC_score','LR_score','GRA_score']].corr(numeric_only=True))


## 7) Ket luan va xuat data cleaned (optional)


In [ ]:
summary = {
    'rows': len(df),
    'band_parse_null_rate': float(df['band_value'].isna().mean()),
    'prompt_dup_rate': float(1 - df['prompt_hash'].nunique()/len(df)),
    'essay_dup_rate': float(1 - df['essay_hash'].nunique()/len(df))
}
summary
